# Churn Prediction using ANN in Pytorch

## 1. Importing Libraries

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

## 2. Loading Dataset

In [46]:
url = "https://media.githubusercontent.com/media/fatahrahimi330/100-Machine-Learning-Projects/refs/heads/master/69-Churn%20Prediction%20ANN%20Pytorch/Churn_Modelling.csv"

In [47]:
df = pd.read_csv(url)
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. Data Preprocessing

In [48]:
df.shape

(10000, 14)

In [49]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RowNumber,10000.0,5.000500e+03,2886.895680,1.00,2500.75,5.000500e+03,7.500250e+03,10000.00
CustomerId,10000.0,1.569094e+07,71936.186123,15565701.00,15628528.25,1.569074e+07,1.575323e+07,15815690.00
CreditScore,10000.0,6.505288e+02,96.653299,350.00,584.00,6.520000e+02,7.180000e+02,850.00
Age,10000.0,3.892180e+01,10.487806,18.00,32.00,3.700000e+01,4.400000e+01,92.00
Tenure,10000.0,5.012800e+00,2.892174,0.00,3.00,5.000000e+00,7.000000e+00,10.00
Balance,10000.0,7.648589e+04,62397.405202,0.00,0.00,9.719854e+04,1.276442e+05,250898.09
NumOfProducts,10000.0,1.530200e+00,0.581654,1.00,1.00,1.000000e+00,2.000000e+00,4.00
HasCrCard,10000.0,7.055000e-01,0.455840,0.00,0.00,1.000000e+00,1.000000e+00,1.00
IsActiveMember,10000.0,5.151000e-01,0.499797,0.00,0.00,1.000000e+00,1.000000e+00,1.00
EstimatedSalary,10000.0,1.000902e+05,57510.492818,11.58,51002.11,1.001939e+05,1.493882e+05,199992.48


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [51]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [52]:
X = df.iloc[:, 3:-1].values
y = df.iloc[:, -1].values

In [53]:
X

array([[619, 'France', 'Female', ..., 1, 1, 101348.88],
       [608, 'Spain', 'Female', ..., 0, 1, 112542.58],
       [502, 'France', 'Female', ..., 1, 0, 113931.57],
       ...,
       [709, 'France', 'Female', ..., 0, 1, 42085.58],
       [772, 'Germany', 'Male', ..., 1, 0, 92888.52],
       [792, 'France', 'Female', ..., 1, 0, 38190.78]], dtype=object)

In [54]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:,2])

In [55]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

transformers = [('encoder', OneHotEncoder(), [1])]
ct = ColumnTransformer(transformers = transformers, remainder = "passthrough")
X = ct.fit_transform(X)
X

array([[1.0, 0.0, 0.0, ..., 1, 1, 101348.88],
       [0.0, 0.0, 1.0, ..., 0, 1, 112542.58],
       [1.0, 0.0, 0.0, ..., 1, 0, 113931.57],
       ...,
       [1.0, 0.0, 0.0, ..., 0, 1, 42085.58],
       [0.0, 1.0, 0.0, ..., 1, 0, 92888.52],
       [1.0, 0.0, 0.0, ..., 1, 0, 38190.78]], dtype=object)

In [56]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [57]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [58]:
import torch
from torch.utils.data import DataLoader, TensorDataset
# 1. Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N, 1)

# 2. DataLoader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

In [59]:
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)  # shape (N, 1)

# 2. DataLoader
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

## 4. Build the Model

In [60]:
import torch
import torch.nn as nn

In [61]:
class ANN(nn.Module):
    def __init__(self, in_features):
        super(ANN, self).__init__()
        self.fc1 = nn.Linear(in_features ,6)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(6, 6)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(6, 1)
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

In [62]:
X_train.shape

torch.Size([8000, 12])

In [63]:
model = ANN(in_features=12)

In [64]:
!pip install torchinfo

In [65]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                   Param #
ANN                                      --
├─Linear: 1-1                            78
├─ReLU: 1-2                              --
├─Linear: 1-3                            42
├─ReLU: 1-4                              --
├─Linear: 1-5                            7
├─Sigmoid: 1-6                           --
Total params: 127
Trainable params: 127
Non-trainable params: 0

In [66]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## 5. Loop and Fit the Model

In [69]:
epochs = 100
# 4. Training Loop
for epoch in range(epochs):
    model.train()
    total_loss    = 0
    total_correct = 0
    total_samples = 0

    for X_batch, y_batch in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        # accuracy
        predicted = (outputs >= 0.5).float()          # threshold at 0.5
        total_correct += (predicted == y_batch).sum().item()
        total_samples += y_batch.size(0)
        total_loss    += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_samples * 100


    print(f"Epoch {epoch+1:3d}/100 | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

100%|██████████| 250/250 [00:00<00:00, 638.64it/s]


Epoch   1/100 | Loss: 0.3337 | Accuracy: 86.10%


100%|██████████| 250/250 [00:00<00:00, 728.72it/s]


Epoch   2/100 | Loss: 0.3336 | Accuracy: 86.21%


100%|██████████| 250/250 [00:00<00:00, 737.31it/s]


...

Epoch  98/100 | Loss: 0.3314 | Accuracy: 86.20%


100%|██████████| 250/250 [00:00<00:00, 477.99it/s]


Epoch  99/100 | Loss: 0.3308 | Accuracy: 86.28%


100%|██████████| 250/250 [00:00<00:00, 483.27it/s]

Epoch 100/100 | Loss: 0.3312 | Accuracy: 86.11%


## 6. Make Prediction

In [71]:
# 1. Scale the input (same scaler from training)
sample = sc.transform([[1,0,0,600,1,40,3,60000,2,1,1,50000]])

# 2. Convert to tensor
sample_tensor = torch.tensor(sample, dtype=torch.float32)

# 3. Predict
model.eval()
with torch.no_grad():
    output = model(sample_tensor)
    result = (output >= 0.5)

print(result.item())  # True or False

False


## 7. Evaluate the Model

In [73]:
all_preds  = []
all_labels = []

# 1. Loop through test loader
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs   = model(X_batch)
        predicted = (outputs >= 0.5).float()

        all_preds.append(predicted)
        all_labels.append(y_batch)

# 2. Concatenate all batches
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

In [74]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

accuracy = accuracy_score(all_labels, all_preds) * 100
cm       = confusion_matrix(all_labels, all_preds)
report   = classification_report(all_labels, all_preds)

print(f"Test Accuracy:  {accuracy:.2f}%")
print(f"Confusion Matrix:\n{cm}")
print(f"Classification Report:\n{report}")

Test Accuracy:  86.25%
Confusion Matrix:
[[1536   71]
 [ 204  189]]
Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.96      0.92      1607
         1.0       0.73      0.48      0.58       393

    accuracy                           0.86      2000
   macro avg       0.80      0.72      0.75      2000
weighted avg       0.85      0.86      0.85      2000

